# P1 · Block 1 — ADLS → Bronze

- **Source:** `raw/orders/`, `raw/customers/` in `stgaccdeprep` (ADLS Gen2, HNS enabled)
- **Target:** `retail.bronze.orders`, `retail.bronze.customers` — Delta, Unity Catalog managed
- **Auth:** UC storage credential → `ac-de-prep` managed identity → RBAC on the storage account. No account keys anywhere in this project.

### Bronze contract

Land the data as it arrives, add provenance, apply nothing else. Duplicate `order_id`s,
null and negative amounts, inconsistent status casing and orphan customer references
are **expected to survive this block** — cleaning is Silver's job.

Rows that fail to parse are captured in `_corrupt_record` rather than dropped, so Bronze
is provably lossless.

In [0]:
STORAGE = "stgaccdeprep"
 
RAW    = f"abfss://raw@{STORAGE}.dfs.core.windows.net"
BRONZE = f"abfss://bronze@{STORAGE}.dfs.core.windows.net"
SILVER = f"abfss://silver@{STORAGE}.dfs.core.windows.net"
GOLD   = f"abfss://gold@{STORAGE}.dfs.core.windows.net"
 
CATALOG = "retail"

### Catalog and schemas

`retail` catalog, one schema per medallion layer.

**`MANAGED LOCATION` matters.** Without it, tables land in the workspace's own storage
and the claim "the medallion lives in my ADLS account" is quietly false.

The catalog needs its own container (`lakehouse`) because UC forbids overlapping managed
locations — catalog root, schema location and table location must not nest.

In [0]:
LAKEHOUSE = f"abfss://lakehouse@{STORAGE}.dfs.core.windows.net"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG} MANAGED LOCATION '{LAKEHOUSE}/'")
spark.sql(f"USE CATALOG {CATALOG}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS bronze MANAGED LOCATION '{BRONZE}/'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS silver MANAGED LOCATION '{SILVER}/'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS gold   MANAGED LOCATION '{GOLD}/'")

display(spark.sql("SHOW SCHEMAS"))

databaseName
bronze
default
gold
information_schema
silver


### Verify path access

Proves the compute can reach the bytes, before adding a read on top.
**403** → RBAC or UC grant. **404** → the path string.

In [0]:
display(dbutils.fs.ls(f"{RAW}/"))

path,name,size,modificationTime
abfss://raw@stgaccdeprep.dfs.core.windows.net/customers/,customers/,0,1788351678000
abfss://raw@stgaccdeprep.dfs.core.windows.net/orders/,orders/,0,1788351670000


In [0]:
display(dbutils.fs.ls(f"{RAW}/orders/"))

path,name,size,modificationTime
abfss://raw@stgaccdeprep.dfs.core.windows.net/orders/orders_batch_01.csv,orders_batch_01.csv,49960,1788351746000


In [0]:
display(dbutils.fs.ls(f"{RAW}/customers/"))

path,name,size,modificationTime
abfss://raw@stgaccdeprep.dfs.core.windows.net/customers/customers_v1.csv,customers_v1.csv,3651,1788351717000


### Explicit schema, not `inferSchema`

Inference costs an extra full pass over the data, guesses types from a sample, and lets a
new file silently change the table's types. A declared schema fails loudly instead.

`order_id` is `STRING` — identifiers are never arithmetic, and `000123` must not lose its
leading zeros. `amount` is `DECIMAL(12,2)`, never `DOUBLE`: binary floating point
accumulates error, which is a reconciliation failure in a financial column.

In [0]:
from pyspark.sql.types import (StructType, StructField, StringType, TimestampType, DecimalType)

orders_schema = StructType([
    StructField("order_id",       StringType(),       True),
    StructField("customer_id",    StringType(),       True),
    StructField("order_ts",       TimestampType(),    True),
    StructField("updated_ts",     TimestampType(),    True),
    StructField("amount",         DecimalType(12, 2), True),
    StructField("status",         StringType(),       True),
    StructField("payment_method", StringType(),       True),
    StructField("_corrupt_record", StringType(),      True),
])

customers_schema = StructType([
    StructField("customer_id",   StringType(),    True),
    StructField("customer_name", StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("segment",       StringType(),    True),
    StructField("country",       StringType(),    True),
    StructField("updated_ts",    TimestampType(), True),
    StructField("_corrupt_record", StringType(),  True),
])

print(orders_schema.simpleString())

struct<order_id:string,customer_id:string,order_ts:timestamp,updated_ts:timestamp,amount:decimal(12,2),status:string,payment_method:string,_corrupt_record:string>


### Read raw

`PERMISSIVE` mode with `_corrupt_record` populated: unparseable rows are **kept and
visible**, not nulled and not fatal. `DROPMALFORMED` would silently discard them —
data loss you'd never detect.

Trade-off: bad rows reach Bronze and Silver has to handle them. Accepted, because
Bronze's contract is that nothing is lost.

In [0]:
orders_raw = (spark.read
    .format("csv")
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(orders_schema)
    .load(f"{RAW}/orders/"))

customers_raw = (spark.read
    .format("csv")
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(customers_schema)
    .load(f"{RAW}/customers/"))

print(f"orders    : {orders_raw.count():>6} rows")
print(f"customers : {customers_raw.count():>6} rows")

orders    :    640 rows
customers :     60 rows


### Add provenance, write Bronze

`_source_file` and `_ingest_ts` answer the only two questions that matter when a
downstream number looks wrong: which file did this row come from, and when did it land.
Without them the question is unanswerable.

`mode("overwrite")` makes this notebook safely re-runnable. `append` would duplicate
every row on a second run — Bronze idempotency here comes from full replacement, not
from a merge key.

In [0]:
from pyspark.sql.functions import col, current_timestamp

orders_bronze = (orders_raw
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingest_ts",   current_timestamp()))

customers_bronze = (customers_raw
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingest_ts",   current_timestamp()))

(orders_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.bronze.orders"))

(customers_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.bronze.customers"))

print("bronze written")

bronze written


### Verify Bronze preserved the defects

Bronze is correct when the mess is still there. Every count below is expected to be
non-zero — if duplicates or nulls came back zero, something cleaned data it shouldn't have.

In [0]:
%sql
SELECT
  count(*)                                                     AS total_rows,
  count(DISTINCT order_id)                                     AS distinct_orders,
  count(*) - count(DISTINCT order_id)                          AS duplicate_rows,
  sum(CASE WHEN amount IS NULL THEN 1 ELSE 0 END)              AS null_amounts,
  sum(CASE WHEN amount <  0    THEN 1 ELSE 0 END)              AS negative_amounts,
  count(DISTINCT status)                                       AS distinct_status,
  sum(CASE WHEN _corrupt_record IS NOT NULL THEN 1 ELSE 0 END) AS corrupt_rows
FROM retail.bronze.orders

total_rows,distinct_orders,duplicate_rows,null_amounts,negative_amounts,distinct_status,corrupt_rows
640,600,40,25,15,12,0


### Transaction log and time travel

`_delta_log` is what turns a directory of Parquet files into a table with ACID guarantees.
It is append-only: `RESTORE` adds a commit, it doesn't erase one.

Direct filesystem access to a managed table's log is **refused** by Unity Catalog —
`PathAuthzRequest ... overlaps with managed storage`. That's the point of managed tables:
the catalog is the only way in, so nobody reads or writes around its access controls.
`DESCRIBE HISTORY` is the supported route.

In [0]:
%sql
DESCRIBE HISTORY retail.bronze.orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-09-02T14:40:01.000Z,148174838415361,k.anushashetty8@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1098799868819359),058f65cc-bc3e-42c4-a5fb-969396208c89,0902-140403-wytjfe1n-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 640, numOutputBytes -> 15550)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13


In [0]:
%sql
DESCRIBE DETAIL retail.bronze.orders

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,e576c66a-46bc-43f7-b8f5-7331f18fd21a,retail.bronze.orders,null,abfss://bronze@stgaccdeprep.dfs.core.windows.net/__unitystorage/schemas/d5fb37a9-53cb-40f3-a6c2-86136e291488/tables/0e1e8c54-cf8c-4567-bfbe-4252960f9fe5,2026-09-02T14:39:58.173Z,2026-09-02T14:40:01.000Z,List(),List(),1,15550,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
INSERT INTO retail.bronze.orders
SELECT * FROM retail.bronze.orders LIMIT 5

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
SELECT
  (SELECT count(*) FROM retail.bronze.orders VERSION AS OF 0) AS v0_rows,
  (SELECT count(*) FROM retail.bronze.orders)                 AS current_rows

v0_rows,current_rows
640,645


In [0]:
%sql
RESTORE TABLE retail.bronze.orders TO VERSION AS OF 0

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
15550,1,1,0,3581,0


In [0]:
%sql
DESCRIBE HISTORY retail.bronze.orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-09-02T15:08:57.000Z,148174838415361,k.anushashetty8@gmail.com,RESTORE,"Map(version -> 0, timestamp -> null)",null,List(1098799868819359),55046756-b8eb-4f20-baa0-6123bbee9c84,0902-140403-wytjfe1n-v2n,1,Serializable,false,"Map(numRestoredFiles -> 0, removedFilesSize -> 3581, numRemovedFiles -> 1, restoredFilesSize -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 15550)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13
1,2026-09-02T15:08:01.000Z,148174838415361,k.anushashetty8@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1098799868819359),774af53e-d15a-4c59-a9ec-a7a694cc8d7f,0902-140403-wytjfe1n-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 3581)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13
0,2026-09-02T14:40:01.000Z,148174838415361,k.anushashetty8@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1098799868819359),058f65cc-bc3e-42c4-a5fb-969396208c89,0902-140403-wytjfe1n-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 640, numOutputBytes -> 15550)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13
